In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, BooleanType, DoubleType, StringType

spark = SparkSession.builder.appName("Steam_Live_Streaming").getOrCreate()

schema = StructType([
    StructField("app_id", IntegerType(), True),
    StructField("helpful", IntegerType(), True),
    StructField("funny", IntegerType(), True),
    StructField("date", StringType(), True), 
    StructField("is_recommended", BooleanType(), True),
    StructField("hours", DoubleType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("review_id", IntegerType(), True)
])

live_df = spark.readStream.schema(schema).option("header", "true").csv("data/streaming_input/")

live_aggregations = live_df.groupBy("is_recommended").count()

print("Starting stream and collecting data into memory...")

query = live_aggregations.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("live_reviews_table") \
    .trigger(processingTime="2 seconds") \
    .start()

time.sleep(10)

print("--- LIVE AGGREGATION RESULTS ---")
spark.sql("SELECT * FROM live_reviews_table").show()

query.stop()
print("Stream stopped safely.")

Starting stream and collecting data into memory...
--- LIVE AGGREGATION RESULTS ---
+--------------+-----+
|is_recommended|count|
+--------------+-----+
|          true|14979|
|         false| 2521|
+--------------+-----+

Stream stopped safely.
